# Planner 

The idea is that the webapp (frontend) sends a request to the backend and 
the backend executes the GenAi component. The result is passed back to the frontend

We will call the message from the UI to Gen as the Payload and the response will be called
Response

This notebook plays with the Planner 

## Imports 

In [16]:

import inspect
import sys

sys.path.append("./")  # Add the parent directory to the Python path
sys.path.append("../")  # Add the parent directory to the Python path
sys.path.append("../../")  # Add the parent directory to the Python path

from langchain_core.messages import HumanMessage, AIMessage, AnyMessage
from langchain_core.tools import StructuredTool

from agentic_system.common.base_task import TaskExecutionContext, TaskResult, TextResult
from agentic_system.common.get_llm import azure_llm_if
from agentic_system.components.results_interpreter.semantic_context import ResultsInterpreterSemanticContext


from langchain.agents import create_agent

#from agentic_system.common.semantic_models import SemanticCatalog 

from dataclasses import dataclass
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
from typing_extensions import Self
from typing import Any, cast

from agentic_system.common.get_llm import azure_llm_if as get_llm
from agentic_system.components.results_interpreter.prompts import RESULTS_INTERPRETER_PROMPT_TEMPLATE
from agentic_system.common.data_component import BaseDataComponent, BaseDomainTools
from agentic_system.common.semantic_models import ColumnCard, SemanticCatalog, SemanticContext, TableCard

from agentic_system.components.planner.planner import * 
from agentic_system.components.results_interpreter.results_interpreter import * 



## Toy example 1

In [ ]:

# from the UI. This is already hooked.  
payload = {
    "subzone": "Select option",
    "project_name": "Demo1",
    "study_name": "Demo1",
    "data_dates": [
        "2020-10-05",
        "2021-11-25"
    ],
    "sectors": [],
    "user_query": 'how many wells are there?'
}

# To do:
# read the real data, process it to the needed shape. 

In [ ]:
# The backend receives the request

planner = get_default_planner()
print( planner.description )


zero temp, seed 42, top_p = 1
Generates a structured execution plan for the user query. 
The plan contains be a list of tasks, each with a unique task_id, an agent responsible for executing it, and any dependencies on other tasks. 
Use this tool to generate a plan. 


In [ ]:
query = payload['user_query']
plan = planner.run( query )



user_intent='Determine the total number of wells in the project.' tasks=[ExecutionTask(instruction='Count the total number of wells in the project, including both injectors and producers.', agent='input_data_analysis', task_id='count_total_wells', depends_on=None)]


In [13]:
plan.model_dump()

{'user_intent': 'Determine the total number of wells in the project.',
 'tasks': [{'instruction': 'Count the total number of wells in the project, including both injectors and producers.',
   'agent': 'input_data_analysis',
   'task_id': 'count_total_wells',
   'depends_on': None}]}

In [ ]:
agent_to_run = plan.tasks[0].agent 
print("The agent to run is -", agent_to_run,"-" )

## Toy example 2

In [14]:
query = "summarize the quality of the simulation results?"
plan = planner.run( query )

plan.model_dump()

{'user_intent': 'Summarize the quality of the simulation results.',
 'tasks': [{'instruction': 'Provide a summary of the quality of the simulation results, including metrics such as history-match quality, model accuracy, and any other relevant indicators.',
   'agent': 'results_interpreter',
   'task_id': 'summarize_simulation_quality',
   'depends_on': None}]}

In [21]:
agent_to_run = plan.tasks[0].agent 
print("The agent to run is -", agent_to_run,"-" )
print("The query is  -", plan.tasks[0].instruction  ,"-" )


The agent to run is - results_interpreter -
The query is  - Provide a summary of the quality of the simulation results, including metrics such as history-match quality, model accuracy, and any other relevant indicators. -


In [23]:
llm = get_llm()
interpreter = ResultsInterpreterComponent( llm = llm )
interpreter.set_data( *prepare_test_data() )


interpreter_response = interpreter.run( plan.tasks[0].instruction )



zero temp, seed 42, top_p = 1
  INJECTOR PRODUCER  PALLOCATION   GAIN  GAIN_CLASS
0       I1       P1     0.508456  0.380  meaningful
1       I2       P1     0.432001  0.300  meaningful
2       I3       P1     0.007279  0.001  negligible
3       I1       P2     0.463442  0.400  meaningful
4       I3       P2     0.226982  0.150  meaningful
  PRODUCER  CORRELATION  VARIANCE_RATIO  QUALITY_SCORE QUALITY_CLASS
0       P1     0.802377        0.929430       0.864767          good
1       P2     0.929300        1.324738       0.886701          good
2       P3     0.754035        1.020565       0.833949          good
3       P4     0.925126        1.205418       0.917859          good
                                                 0      1      2       3
PRODUCER                                        P1     P2     P3      P4
current_produced_water_fraction               0.72   0.35   0.88     0.2
current_produced_oil_fraction                 0.28   0.65   0.12     0.8
current_volume_liquid

In [35]:
interpreter_response.model_dump()

print( "The type of response is ", type(interpreter_response) )
print( "Properties of the object are several... but two stand out ")

# what kind of data (artifact) was produced?
print( type(interpreter_response.data_results))       # it is always a list 
print( type(interpreter_response.data_results[0]))    # the element as "text" or "dataframes"

print( 100 * '=')
if isinstance( interpreter_response.data_results[0] , TextResult ):
    print("We have a response of the type: Text, we can just pass it to the presenter")
    print( interpreter_response.data_results[0].text)


The type of response is  <class 'agentic_system.common.base_task.TaskResult'>
Properties of the object are several... but two stand out 
<class 'list'>
<class 'agentic_system.common.base_task.TextResult'>
We have a response of the type: Text, we can just pass it to the presenter
The simulation results indicate a generally high quality of history matching and model accuracy across the producers. Here is a summary of the key metrics:

1. **Overall Quality**:
   - All producers are classified as having "good" quality based on the composite quality score.

2. **Correlation**:
   - The correlation between observed and simulated liquid production is strong for all producers, ranging from 0.754 (P3) to 0.929 (P2). This indicates a good alignment between the modeled and observed production trends.

3. **Variance Ratio**:
   - The variance ratio, which compares the simulated production variance to the observed variance, is close to 1 for most producers, indicating balanced variance matching. P2

And from here, the results of all the tasks are stored in the graph.
The presenter receive those task results and produced an output for the UI





## Note: 

So basically, we can just use a for loop
but thenm if we wanted to have "memory" of the conversation, we couldnt.
Everything is lost after the planner-interpreter runs. 
Thats where the langchain Graph enters into play.
The graph has a "state" and the state can be re-used. 




## To do:

Lets get an end-to-end interpreter including the presenter, a mock of the payload, a mock of the data.

First get it working in a notebook. Then get it done in a simple UI that just displays HTML charts + text 

So...


1. Decide how to pre-process the data and put it in the way we -believe- we need
2. For that, be able to read from disk (using the lib) and create the tables.
3. Load (or hard-code) initilaly the semantic model.
4. Create the entire interpreter component
5. Run the interpreter with a Fake instruction
6. Run it with the output of the planner 
7. Pass the results to the Presenter.
8. Presenter to produce an HTML-friendly output
9. Basic CSS, HTML, JS webapp displays the results 


## Toy 3
will be used to develop the presenter


In [37]:
from agentic_system.components.direct_answer import *
from agentic_system.components.planner import *
from agentic_system.components.results_interpreter import *


In [ ]:
 
planner = get_default_planner()
print( planner.description )


direct_answer = get_default_direct_answer( llm =get_llm()) 

query= "Explain to me  whats VRR? Then also tell me whats waterflood in reservopir engineering"
plan = planner.run( query )
plan.model_dump() 


zero temp, seed 42, top_p = 1
Generates a structured execution plan for the user query. 
The plan contains be a list of tasks, each with a unique task_id, an agent responsible for executing it, and any dependencies on other tasks. 
Use this tool to generate a plan. 
zero temp, seed 42, top_p = 1


{'user_intent': 'Understand the concept of VRR and waterflooding in reservoir engineering.',
 'tasks': [{'instruction': 'Explain the concept of VRR (Voidage Replacement Ratio) in reservoir engineering.',
   'agent': 'direct_answer',
   'task_id': 'explain_vrr_concept',
   'depends_on': None},
  {'instruction': 'Explain the concept of waterflooding in reservoir engineering.',
   'agent': 'direct_answer',
   'task_id': 'explain_waterflooding_concept',
   'depends_on': None}]}

In [55]:
instruction1 = plan.tasks[0].instruction
task_result1 = direct_answer.run( instruction1  )

instruction2 = plan.tasks[1].instruction
task_result2 = direct_answer.run( instruction2  )

# the presenter will receive this at the end 
task_results = [ task_result1, task_result2 ]


In [ ]:
task_results

TaskResult

[TaskResult(agent='direct_answer', instruction='Explain the concept of VRR (Voidage Replacement Ratio) in reservoir engineering.', cheap_output='The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of injected fluids}}{\\text{Volume of produced fluids}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\n3. **Implications**:\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\n   - **VRR > 1**: Indicates over-injection, which m

In [76]:
print( task_result1.model_dump())



from pydantic import TypeAdapter

adapter = TypeAdapter(list[TaskResult])

# list[TaskResult] → JSON string
txt = adapter.dump_json(task_results).decode("utf-8")

# JSON string → list[TaskResult]
#restored_task_results = adapter.validate_json(txt)
print(repr(txt))

{'agent': 'direct_answer', 'instruction': 'Explain the concept of VRR (Voidage Replacement Ratio) in reservoir engineering.', 'cheap_output': 'The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of injected fluids}}{\\text{Volume of produced fluids}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\n3. **Implications**:\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\n   - **VRR > 1**: Indicates over-injection, which may

In [77]:
print(repr(txt))

'[{"agent":"direct_answer","instruction":"Explain the concept of VRR (Voidage Replacement Ratio) in reservoir engineering.","cheap_output":"The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\\n\\n\\\\[\\n\\\\text{VRR} = \\\\frac{\\\\text{Volume of injected fluids}}{\\\\text{Volume of produced fluids}}\\n\\\\]\\n\\n### Key Points:\\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\\n3. **Implications**:\\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\\n   - **VRR > 1**: Indicates over-

In [78]:
print( txt )
s = '[{"agent":"direct_answer","instruction":"Explain the concept of VRR (Voidage Replacement Ratio) in reservoir engineering.","cheap_output":"The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\\n\\n\\\\[\\n\\\\text{VRR} = \\\\frac{\\\\text{Volume of injected fluids}}{\\\\text{Volume of produced fluids}}\\n\\\\]\\n\\n### Key Points:\\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\\n3. **Implications**:\\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\\n   - **VRR > 1**: Indicates over-injection, which may cause operational issues like fracturing or inefficient sweep.\\n\\nMaintaining an appropriate VRR is critical for optimizing recovery, managing reservoir pressure, and ensuring long-term reservoir performance.","raw_results":["The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\\n\\n\\\\[\\n\\\\text{VRR} = \\\\frac{\\\\text{Volume of injected fluids}}{\\\\text{Volume of produced fluids}}\\n\\\\]\\n\\n### Key Points:\\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\\n3. **Implications**:\\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\\n   - **VRR > 1**: Indicates over-injection, which may cause operational issues like fracturing or inefficient sweep.\\n\\nMaintaining an appropriate VRR is critical for optimizing recovery, managing reservoir pressure, and ensuring long-term reservoir performance."],"data_results":[{"text":"The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\\n\\n\\\\[\\n\\\\text{VRR} = \\\\frac{\\\\text{Volume of injected fluids}}{\\\\text{Volume of produced fluids}}\\n\\\\]\\n\\n### Key Points:\\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\\n3. **Implications**:\\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\\n   - **VRR > 1**: Indicates over-injection, which may cause operational issues like fracturing or inefficient sweep.\\n\\nMaintaining an appropriate VRR is critical for optimizing recovery, managing reservoir pressure, and ensuring long-term reservoir performance.","role":"answer"}]},{"agent":"direct_answer","instruction":"Explain the concept of waterflooding in reservoir engineering.","cheap_output":"Waterflooding is a secondary recovery technique in reservoir engineering used to enhance oil production after the natural reservoir drive has declined. It involves injecting water into the reservoir through dedicated injection wells to maintain reservoir pressure and displace oil toward production wells. \\n\\nThe process works by increasing the sweep efficiency, both areally and vertically, pushing the oil trapped in the pore spaces toward producing wells. Waterflooding is particularly effective in reservoirs with good permeability and connectivity. Key considerations include the water injection rate, pattern design (e.g., five-spot, seven-spot), and the mobility ratio between water and oil to ensure efficient displacement.","raw_results":["Waterflooding is a secondary recovery technique in reservoir engineering used to enhance oil production after the natural reservoir drive has declined. It involves injecting water into the reservoir through dedicated injection wells to maintain reservoir pressure and displace oil toward production wells. \\n\\nThe process works by increasing the sweep efficiency, both areally and vertically, pushing the oil trapped in the pore spaces toward producing wells. Waterflooding is particularly effective in reservoirs with good permeability and connectivity. Key considerations include the water injection rate, pattern design (e.g., five-spot, seven-spot), and the mobility ratio between water and oil to ensure efficient displacement."],"data_results":[{"text":"Waterflooding is a secondary recovery technique in reservoir engineering used to enhance oil production after the natural reservoir drive has declined. It involves injecting water into the reservoir through dedicated injection wells to maintain reservoir pressure and displace oil toward production wells. \\n\\nThe process works by increasing the sweep efficiency, both areally and vertically, pushing the oil trapped in the pore spaces toward producing wells. Waterflooding is particularly effective in reservoirs with good permeability and connectivity. Key considerations include the water injection rate, pattern design (e.g., five-spot, seven-spot), and the mobility ratio between water and oil to ensure efficient displacement.","role":"answer"}]}]'
restored_task_results = adapter.validate_json(s)
restored_task_results


[{"agent":"direct_answer","instruction":"Explain the concept of VRR (Voidage Replacement Ratio) in reservoir engineering.","cheap_output":"The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of injected fluids}}{\\text{Volume of produced fluids}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\n3. **Implications**:\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\n   - **VRR > 1**: Indicates over-injection, which may cau

[TaskResult(agent='direct_answer', instruction='Explain the concept of VRR (Voidage Replacement Ratio) in reservoir engineering.', cheap_output='The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of injected fluids}}{\\text{Volume of produced fluids}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\n3. **Implications**:\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\n   - **VRR > 1**: Indicates over-injection, which m

In [72]:
s = """[{"agent":"direct_answer","instruction":"Explain the concept of VRR (Voidage Replacement Ratio) in reservoir engineering.","cheap_output":"The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of injected fluids}}{\\text{Volume of produced fluids}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\n3. **Implications**:\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\n   - **VRR > 1**: Indicates over-injection, which may cause operational issues like fracturing or inefficient sweep.\n\nMaintaining an appropriate VRR is critical for optimizing recovery, managing reservoir pressure, and ensuring long-term reservoir performance.","raw_results":["The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of injected fluids}}{\\text{Volume of produced fluids}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\n3. **Implications**:\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\n   - **VRR > 1**: Indicates over-injection, which may cause operational issues like fracturing or inefficient sweep.\n\nMaintaining an appropriate VRR is critical for optimizing recovery, managing reservoir pressure, and ensuring long-term reservoir performance."],"data_results":[{"text":"The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced from it. It is defined as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of injected fluids}}{\\text{Volume of produced fluids}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess reservoir pressure maintenance and the effectiveness of secondary recovery methods, such as waterflooding or gas injection.\n2. **Ideal Value**: A VRR of 1.0 indicates that the volume of injected fluids equals the volume of produced fluids, which helps maintain reservoir pressure.\n3. **Implications**:\n   - **VRR < 1**: Indicates insufficient injection, leading to pressure depletion and potentially reduced recovery.\n   - **VRR > 1**: Indicates over-injection, which may cause operational issues like fracturing or inefficient sweep.\n\nMaintaining an appropriate VRR is critical for optimizing recovery, managing reservoir pressure, and ensuring long-term reservoir performance.","role":"answer"}]},{"agent":"direct_answer","instruction":"Explain the concept of waterflooding in reservoir engineering.","cheap_output":"Waterflooding is a secondary recovery technique in reservoir engineering used to enhance oil production after the natural reservoir drive has declined. It involves injecting water into the reservoir through dedicated injection wells to maintain reservoir pressure and displace oil toward production wells. \n\nThe process works by increasing the sweep efficiency, both areally and vertically, pushing the oil trapped in the pore spaces toward producing wells. Waterflooding is particularly effective in reservoirs with good permeability and connectivity. Key considerations include the water injection rate, pattern design (e.g., five-spot, seven-spot), and the mobility ratio between water and oil to ensure efficient displacement.","raw_results":["Waterflooding is a secondary recovery technique in reservoir engineering used to enhance oil production after the natural reservoir drive has declined. It involves injecting water into the reservoir through dedicated injection wells to maintain reservoir pressure and displace oil toward production wells. \n\nThe process works by increasing the sweep efficiency, both areally and vertically, pushing the oil trapped in the pore spaces toward producing wells. Waterflooding is particularly effective in reservoirs with good permeability and connectivity. Key considerations include the water injection rate, pattern design (e.g., five-spot, seven-spot), and the mobility ratio between water and oil to ensure efficient displacement."],"data_results":[{"text":"Waterflooding is a secondary recovery technique in reservoir engineering used to enhance oil production after the natural reservoir drive has declined. It involves injecting water into the reservoir through dedicated injection wells to maintain reservoir pressure and displace oil toward production wells. \n\nThe process works by increasing the sweep efficiency, both areally and vertically, pushing the oil trapped in the pore spaces toward producing wells. Waterflooding is particularly effective in reservoirs with good permeability and connectivity. Key considerations include the water injection rate, pattern design (e.g., five-spot, seven-spot), and the mobility ratio between water and oil to ensure efficient displacement.","role":"answer"}]}] <class 'str'>"""
restored_task_results = adapter.validate_json(s)
restored_task_results

ValidationError: 1 validation error for list[TaskResult]
  Invalid JSON: control character (\u0000-\u001F) found while parsing a string at line 2 column 0 [type=json_invalid, input_value='[{"agent":"direct_answer...er"}]}] <class \'str\'>', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid

In [58]:
json_string = """{'agent': 'direct_answer', 'instruction': 'Explain the concept of VRR (Voidage Replacement Ratio) in reservoir engineering.', 'cheap_output': 'The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the efficiency of fluid injection in maintaining reservoir pressure during production. It is defined as the ratio of the volume of injected fluid (e.g., water, gas, or other substances) to the volume of fluids produced (oil, gas, and water) over a specific time period. Mathematically, it is expressed as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of Injected Fluid}}{\\text{Volume of Produced Fluid}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess whether the injected fluid is adequately replacing the voidage created by the production of hydrocarbons and associated fluids. Maintaining a VRR close to 1.0 is often desirable to sustain reservoir pressure and optimize recovery.\n\n2. **VRR < 1**: Indicates that the reservoir is under-injected, leading to pressure depletion, which can reduce recovery efficiency and potentially cause issues like gas cap expansion or water coning.\n\n3. **VRR > 1**: Suggests over-injection, which can lead to operational inefficiencies, increased costs, and potential damage to the reservoir, such as fracturing or bypassing hydrocarbons.\n\n4. **Applications**: VRR is critical in waterflooding, gas injection, and other enhanced oil recovery (EOR) methods to ensure effective reservoir management and maximize hydrocarbon recovery.\n\nBy monitoring and managing VRR, reservoir engineers can optimize production strategies and maintain reservoir performance over the life of the field.', 'raw_results': ['The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the efficiency of fluid injection in maintaining reservoir pressure during production. It is defined as the ratio of the volume of injected fluid (e.g., water, gas, or other substances) to the volume of fluids produced (oil, gas, and water) over a specific time period. Mathematically, it is expressed as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of Injected Fluid}}{\\text{Volume of Produced Fluid}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess whether the injected fluid is adequately replacing the voidage created by the production of hydrocarbons and associated fluids. Maintaining a VRR close to 1.0 is often desirable to sustain reservoir pressure and optimize recovery.\n\n2. **VRR < 1**: Indicates that the reservoir is under-injected, leading to pressure depletion, which can reduce recovery efficiency and potentially cause issues like gas cap expansion or water coning.\n\n3. **VRR > 1**: Suggests over-injection, which can lead to operational inefficiencies, increased costs, and potential damage to the reservoir, such as fracturing or bypassing hydrocarbons.\n\n4. **Applications**: VRR is critical in waterflooding, gas injection, and other enhanced oil recovery (EOR) methods to ensure effective reservoir management and maximize hydrocarbon recovery.\n\nBy monitoring and managing VRR, reservoir engineers can optimize production strategies and maintain reservoir performance over the life of the field.'], 'data_results': [{'text': 'The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the efficiency of fluid injection in maintaining reservoir pressure during production. It is defined as the ratio of the volume of injected fluid (e.g., water, gas, or other substances) to the volume of fluids produced (oil, gas, and water) over a specific time period. Mathematically, it is expressed as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of Injected Fluid}}{\\text{Volume of Produced Fluid}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess whether the injected fluid is adequately replacing the voidage created by the production of hydrocarbons and associated fluids. Maintaining a VRR close to 1.0 is often desirable to sustain reservoir pressure and optimize recovery.\n\n2. **VRR < 1**: Indicates that the reservoir is under-injected, leading to pressure depletion, which can reduce recovery efficiency and potentially cause issues like gas cap expansion or water coning.\n\n3. **VRR > 1**: Suggests over-injection, which can lead to operational inefficiencies, increased costs, and potential damage to the reservoir, such as fracturing or bypassing hydrocarbons.\n\n4. **Applications**: VRR is critical in waterflooding, gas injection, and other enhanced oil recovery (EOR) methods to ensure effective reservoir management and maximize hydrocarbon recovery.\n\nBy monitoring and managing VRR, reservoir engineers can optimize production strategies and maintain reservoir performance over the life of the field.', 'role': 'answer'}]}
{"agent": "direct_answer", "instruction": "Explain the concept of VRR (Voidage Replacement Ratio) in reservoir engineering.", "cheap_output": "The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the efficiency of fluid injection in maintaining reservoir pressure during production. It is defined as the ratio of the volume of injected fluid (e.g., water, gas, or other substances) to the volume of fluids produced (oil, gas, and water) over a specific time period. Mathematically, it is expressed as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of Injected Fluid}}{\\text{Volume of Produced Fluid}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess whether the injected fluid is adequately replacing the voidage created by the production of hydrocarbons and associated fluids. Maintaining a VRR close to 1.0 is often desirable to sustain reservoir pressure and optimize recovery.\n\n2. **VRR < 1**: Indicates that the reservoir is under-injected, leading to pressure depletion, which can reduce recovery efficiency and potentially cause issues like gas cap expansion or water coning.\n\n3. **VRR > 1**: Suggests over-injection, which can lead to operational inefficiencies, increased costs, and potential damage to the reservoir, such as fracturing or bypassing hydrocarbons.\n\n4. **Applications**: VRR is critical in waterflooding, gas injection, and other enhanced oil recovery (EOR) methods to ensure effective reservoir management and maximize hydrocarbon recovery.\n\nBy monitoring and managing VRR, reservoir engineers can optimize production strategies and maintain reservoir performance over the life of the field.", "raw_results": ["The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the efficiency of fluid injection in maintaining reservoir pressure during production. It is defined as the ratio of the volume of injected fluid (e.g., water, gas, or other substances) to the volume of fluids produced (oil, gas, and water) over a specific time period. Mathematically, it is expressed as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of Injected Fluid}}{\\text{Volume of Produced Fluid}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess whether the injected fluid is adequately replacing the voidage created by the production of hydrocarbons and associated fluids. Maintaining a VRR close to 1.0 is often desirable to sustain reservoir pressure and optimize recovery.\n\n2. **VRR < 1**: Indicates that the reservoir is under-injected, leading to pressure depletion, which can reduce recovery efficiency and potentially cause issues like gas cap expansion or water coning.\n\n3. **VRR > 1**: Suggests over-injection, which can lead to operational inefficiencies, increased costs, and potential damage to the reservoir, such as fracturing or bypassing hydrocarbons.\n\n4. **Applications**: VRR is critical in waterflooding, gas injection, and other enhanced oil recovery (EOR) methods to ensure effective reservoir management and maximize hydrocarbon recovery.\n\nBy monitoring and managing VRR, reservoir engineers can optimize production strategies and maintain reservoir performance over the life of the field."], "data_results": [{"text": "The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the efficiency of fluid injection in maintaining reservoir pressure during production. It is defined as the ratio of the volume of injected fluid (e.g., water, gas, or other substances) to the volume of fluids produced (oil, gas, and water) over a specific time period. Mathematically, it is expressed as:\n\n\\[\n\\text{VRR} = \\frac{\\text{Volume of Injected Fluid}}{\\text{Volume of Produced Fluid}}\n\\]\n\n### Key Points:\n1. **Purpose**: VRR is used to assess whether the injected fluid is adequately replacing the voidage created by the production of hydrocarbons and associated fluids. Maintaining a VRR close to 1.0 is often desirable to sustain reservoir pressure and optimize recovery.\n\n2. **VRR < 1**: Indicates that the reservoir is under-injected, leading to pressure depletion, which can reduce recovery efficiency and potentially cause issues like gas cap expansion or water coning.\n\n3. **VRR > 1**: Suggests over-injection, which can lead to operational inefficiencies, increased costs, and potential damage to the reservoir, such as fracturing or bypassing hydrocarbons.\n\n4. **Applications**: VRR is critical in waterflooding, gas injection, and other enhanced oil recovery (EOR) methods to ensure effective reservoir management and maximize hydrocarbon recovery.\n\nBy monitoring and managing VRR, reservoir engineers can optimize production strategies and maintain reservoir performance over the life of the field.", "role": "answer"}]}"""


from pydantic import TypeAdapter

task_results_adapter = TypeAdapter(list[TaskResult])

# JSON string → list[TaskResult]
task_results = task_results_adapter.validate_json(txt)

ValidationError: 1 validation error for list[TaskResult]
  Input should be a valid array [type=list_type, input_value={'agent': 'direct_answer'...d.', 'role': 'answer'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

In [ ]:
task_results

# Build the presenter.

The idea, you get a list of "TaskResult" items. Each agent produces one.
Each task result contains one or more TaskItems, the items are of basically 2 types:
1. Text
2. DataFrame (or table)

The presenter converts them to something you can display in HTML
Depending on the query, etc.... a table could be presented as a table.
It could also be presented as a chart. It is the work of the presenter to 
produce something that directly can be placed into HTML 


